# Data Quality and Testing

Notebook for **Chapter 5** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 The Dimensions of Quality and Basic Checks

In [ ]:
import sqlite3, datetime
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE orders(order_id TEXT, customer_id TEXT, amount REAL, status TEXT, created_at TEXT);
INSERT INTO orders VALUES
('O1','C1',250.0,'PAID','2024-06-01T10:00:00'),
('O2','C2',-5.0,'PAID','2024-06-01T11:00:00'),
('O2','C2',80.0,'PAID','2024-06-01T11:05:00'),
('O3',NULL,120.0,'SHIPPED','2024-06-01T12:00:00'),
('O4','C9',99.0,'UNKNOWN','2024-05-20T12:00:00');
""")
def check(name, sql):
    bad = db.execute(sql).fetchone()[0]
    print(f"{'PASS' if bad == 0 else 'FAIL'}  {name:28s} bad_rows={bad}")
    return bad == 0
results = [
 check("unique order_id",        "SELECT COUNT(*)-COUNT(DISTINCT order_id) FROM orders"),
 check("customer_id not null",   "SELECT COUNT(*) FROM orders WHERE customer_id IS NULL"),
 check("amount >= 0",            "SELECT COUNT(*) FROM orders WHERE amount<0"),
 check("status in allowed set",  "SELECT COUNT(*) FROM orders WHERE status NOT IN ('PAID','SHIPPED','REFUNDED')"),
]
now = datetime.datetime(2024, 6, 2, 9, 0, 0)
newest = datetime.datetime.fromisoformat(db.execute("SELECT MAX(created_at) FROM orders").fetchone()[0])
age = (now - newest).total_seconds() / 3600
print(f"{'PASS' if age < 24 else 'FAIL'}  freshness (<24h)             age_hours={age:.1f}")
history = [1000, 980, 1020, 1010, 990]; today = 612
avg = sum(history) / len(history); ratio = today / avg
print(f"{'PASS' if 0.7 <= ratio <= 1.3 else 'FAIL'}  volume within 70-130% of avg  ratio={ratio:.2f}")
expected = {"order_id":"TEXT","customer_id":"TEXT","amount":"REAL","status":"TEXT","created_at":"TEXT"}
actual = {r[1]: r[2] for r in db.execute("PRAGMA table_info(orders)")}
print("PASS  schema matches" if actual == expected else f"FAIL  schema diff {set(expected.items()) ^ set(actual.items())}")
print("overall:", "OK" if all(results) else "BLOCK the pipeline (fail loudly)")

## 5.2 Data Contracts, Schema Drift and Quarantine

In [ ]:
CONTRACT = {"order_id": {"type": str, "required": True},
            "amount_paise": {"type": int, "required": True, "min": 0},
            "status": {"type": str, "required": True, "allowed": {"PAID", "SHIPPED", "REFUNDED"}},
            "coupon": {"type": str, "required": False}}
def validate(rec):
    errs = []
    for f, rule in CONTRACT.items():
        if f not in rec or rec[f] is None:
            if rule["required"]: errs.append(f"{f}: missing")
            continue
        v = rec[f]
        if not isinstance(v, rule["type"]): errs.append(f"{f}: expected {rule['type'].__name__}, got {type(v).__name__}")
        elif "min" in rule and v < rule["min"]: errs.append(f"{f}: {v} < {rule['min']}")
        elif "allowed" in rule and v not in rule["allowed"]: errs.append(f"{f}: {v!r} not allowed")
    extra = set(rec) - set(CONTRACT)
    if extra: errs.append(f"unexpected fields (schema drift): {sorted(extra)}")
    return errs
batch = [{"order_id": "O1", "amount_paise": 25000, "status": "PAID"},
         {"order_id": "O2", "amount_paise": "1,250.50", "status": "PAID"},
         {"order_id": "O3", "amount_paise": -5, "status": "LOST"},
         {"order_id": "O4", "amount_paise": 900, "status": "SHIPPED", "gift_wrap": True},
         {"amount_paise": 100, "status": "PAID"}]
ok = 0
for r in batch:
    e = validate(r); ok += not e
    print("OK  " if not e else "FAIL", r.get("order_id", "<none>"), e)
print(f"valid {ok}/{len(batch)} = {ok/len(batch):.0%}; threshold 95% -> {'PASS' if ok/len(batch) >= 0.95 else 'FAIL the load'}")

## 5.3 Testing Transformations and Tools of the Trade

In [ ]:
import unittest
def clean_amount(raw):
    """paise string -> rupees float, or None if invalid"""
    try: v = int(raw)
    except (TypeError, ValueError): return None
    return None if v < 0 else v / 100
def split_valid(rows):
    good, bad = [], []
    for r in rows:
        a = clean_amount(r.get("amount_paise"))
        (bad if a is None or not r.get("order_id") else good).append(
            {**r, "amount": a} if a is not None else {**r, "reason": "bad amount or id"})
    return good, bad
class T(unittest.TestCase):
    def test_ok(self): self.assertEqual(clean_amount("25050"), 250.5)
    def test_negative(self): self.assertIsNone(clean_amount("-5"))
    def test_text(self): self.assertIsNone(clean_amount("abc"))
    def test_none(self): self.assertIsNone(clean_amount(None))
    def test_split(self):
        g, b = split_valid([{"order_id": "1", "amount_paise": "100"}, {"order_id": "2", "amount_paise": "x"},
                            {"order_id": "", "amount_paise": "5"}])
        self.assertEqual((len(g), len(b)), (1, 2))
if __name__ == "__main__": unittest.main(verbosity=2)

In [ ]:
# pandera (concept)
import pandera as pa
schema = pa.DataFrameSchema({
    "order_id": pa.Column(str, unique=True),
    "amount":   pa.Column(float, pa.Check.ge(0)),
    "status":   pa.Column(str, pa.Check.isin(["PAID", "SHIPPED", "REFUNDED"])),
})
validated = schema.validate(df)        # raises SchemaError listing failures

## 5.4 Observability: SLOs, Anomaly Detection and Incident Reports

In [ ]:
import numpy as np
rng = np.random.default_rng(5)
pattern = [1000, 1000, 1000, 1000, 1000, 400, 400]            # Mon..Sun
history = [int(round(pattern[d % 7] + rng.normal(0, 30))) for d in range(28)]   # 4 weeks, starts on Monday
today = 700                                                    # a Monday (day 28)

h = np.array(history)
z_plain = (today - h.mean()) / h.std(ddof=1)
same_weekday = h[0::7]                                         # the four earlier Mondays
z_seasonal = (today - same_weekday.mean()) / same_weekday.std(ddof=1)
print("history mean %.1f  sd %.1f" % (h.mean(), h.std(ddof=1)))
print("earlier Mondays:", same_weekday.tolist())
print("plain z    = %.2f -> %s" % (z_plain, "FLAG" if abs(z_plain) > 3 else "ok"))
print("seasonal z = %.2f -> %s" % (z_seasonal, "FLAG" if abs(z_seasonal) > 3 else "ok"))

history mean 822.8  sd 280.7
earlier Mondays: [976, 976, 1006, 1017]
plain z    = -0.44 -> ok
seasonal z = -14.00 -> FLAG


In [ ]:
import numpy as np
history = np.array([1000, 980, 1020, 1010, 990, 100, 1005, 995, 1015, 985])   # day 6 was an outage
today = 700
mean, sd = history.mean(), history.std(ddof=1)
z = (today - mean) / sd
median = np.median(history)
mad = np.median(np.abs(history - median))
mz = 0.6745 * (today - median) / mad
print("mean %.1f  sd %.1f  median %.1f  MAD %.1f" % (mean, sd, median, mad))
print("plain z    = %.2f -> %s" % (z, "FLAG" if abs(z) > 3 else "ok"))
print("modified z = %.2f -> %s" % (mz, "FLAG" if abs(mz) > 3.5 else "ok"))

mean 910.0  sd 284.9  median 997.5  MAD 12.5
plain z    = -0.74 -> ok
modified z = -16.05 -> FLAG


## Chapter recap

## Practice set

**1.** (easy) What does `COUNT(*) − COUNT(DISTINCT order_id)` measure? **Answer:** the number of extra duplicate rows for `order_id`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch05.html